In [12]:
# ============================================
# 0. Imports & Spark session
# ============================================
import time
import builtins  # <-- IMPORTANT
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    avg,
    round as spark_round,   # Spark round ONLY for Columns
    count,
    col,
    sum as _sum
)

spark = (
    SparkSession.builder
    .appName("PostgresVsSparkBenchmark")
    .config("spark.jars.packages", "org.postgresql:postgresql:42.7.2")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "/tmp/spark-events")
    .config("spark.history.fs.logDirectory", "/tmp/spark-events")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.default.parallelism", "4")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")


In [13]:
# ============================================
# 1. JDBC connection config
# ============================================

jdbc_url = "jdbc:postgresql://postgres:5432/postgres"
jdbc_props = {
    "user": "postgres",
    "password": "postgres",
    "driver": "org.postgresql.Driver"
}

In [14]:
# ============================================
# 2. Load data from PostgreSQL
# ============================================

print("\n=== Loading orders_big from PostgreSQL ===")

start = time.time()

df_big = spark.read.jdbc(
    url=jdbc_url,
    table="orders_big",
    properties=jdbc_props
)

# Force materialization
row_count = df_big.count()

print(f"Rows loaded: {row_count}")
print("Load time:", builtins.round(time.time() - start, 2), "seconds")

# Register temp view
df_big.createOrReplaceTempView("orders_big")


=== Loading orders_big from PostgreSQL ===
Rows loaded: 1000000
Load time: 0.9 seconds


In [15]:
# ============================================
# 3. Query (a): What is the single item with the highest price_per_category?
# ============================================

print("\n=== Query (a): What is the single item with the highest price_per_category? ===")

start = time.time()

q_a = (
    df_big
    .orderBy("price_per_category", ascending=False)
    .limit(1)
)

q_a.collect()
q_a.show(truncate=False)
print("Query (a) time:", builtins.round(time.time() - start, 2), "seconds")


=== Query (a): What is the single item with the highest price_per_category? ===
+------+-------------+----------------+--------+------------------+----------+-------+
|id    |customer_name|product_category|quantity|price_per_category|order_date|country|
+------+-------------+----------------+--------+------------------+----------+-------+
|841292|Emma Brown   |Automotive      |3       |2000.0            |2024-10-11|Italy  |
+------+-------------+----------------+--------+------------------+----------+-------+

Query (a) time: 10.02 seconds


In [16]:
# ============================================
# 4. Query (b): What are the top 3 products with the highest total quantity sold across all orders?
# ============================================

print("\n=== Query (b): What are the top 3 products with the highest total quantity sold across all orders? ===")

start = time.time()

q_b = ( 
    df_big
    .groupBy("product_category")
    .agg(_sum("quantity").alias("total_quantity"))
    .orderBy(col("total_quantity").desc())
    .limit(3)
)

q_b.collect()
q_b.show(truncate=False)
print("Query (b) time:", builtins.round(time.time() - start, 2), "seconds")


=== Query (b): What are the top 3 products with the highest total quantity sold across all orders? ===
+----------------+--------------+
|product_category|total_quantity|
+----------------+--------------+
|Health & Beauty |300842        |
|Electronics     |300804        |
|Toys            |300598        |
+----------------+--------------+

Query (b) time: 3.0 seconds


In [17]:
# ============================================
# 5. Query (c): What is the total revenue per product category?
# ============================================

print("\n=== Query (c): What is the total revenue per product category? ===")

start = time.time()

q_c = spark.sql("""SELECT product_category, SUM(revenue) FROM (SELECT product_category, (quantity * price_per_category) as revenue FROM orders_big) sub GROUP BY product_category;""")
                
q_c.collect()
q_c.show(truncate=False)
print("Query (c) time:", builtins.round(time.time() - start, 2), "seconds")


=== Query (c): What is the total revenue per product category? ===
+----------------+--------------------+
|product_category|sum(revenue)        |
+----------------+--------------------+
|Home & Garden   |7.80237800900001E7  |
|Books           |1.273197603999989E7 |
|Automotive      |3.065897988599943E8 |
|Electronics     |2.4152500945000267E8|
|Office Supplies |3.8276061640000574E7|
|Grocery         |1.5268355660000028E7|
|Health & Beauty |4.65998178900003E7  |
|Toys            |2.3271039019999716E7|
|Fashion         |3.1566368219999947E7|
|Sports          |6.1848990830000326E7|
+----------------+--------------------+

Query (c) time: 3.26 seconds


In [18]:
# ============================================
# 6. Query (d): Which customers have the highest total spending?
# ============================================

print("\n=== Query (d): Top 10 customers by total spending ===")

start = time.time()

q_d = spark.sql("""
SELECT customer_name, SUM(quantity * price_per_category) AS total_spending
FROM orders_big
GROUP BY customer_name
ORDER BY total_spending DESC
LIMIT 10
""")

q_d.collect()
q_d.show(truncate=False)
print("Query (d) time:", builtins.round(time.time() - start, 2), "seconds")


=== Query (d): Top 10 customers by total spending ===
+--------------+-----------------+
|customer_name |total_spending   |
+--------------+-----------------+
|Carol Taylor  |991179.1800000003|
|Nina Lopez    |975444.9499999998|
|Daniel Jackson|959344.4800000001|
|Carol Lewis   |947708.5700000002|
|Daniel Young  |946030.1400000004|
|Alice Martinez|935100.0199999999|
|Ethan Perez   |934841.2399999991|
|Leo Lee       |934796.4799999993|
|Eve Young     |933176.8599999989|
|Ivy Rodriguez |925742.6400000005|
+--------------+-----------------+

Query (d) time: 3.73 seconds


In [19]:
# ============================================
# 7. Cleanup
# ============================================

spark.stop()